# Optic disc detection using YOLO

In [ ]:
import os
import io
import math
import cv2
import numpy as np
from tqdm import tqdm
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects
from ultralytics import YOLO


In [ ]:
def show_optic_disc_shapes(model, image_dir, image_numbers, conf=0.25, cols=5, figsize=(20, 10)):
    """
    Predicts segmentation masks for multiple images and displays
    the exact predicted mask contours (borders) for each image.

    Args:
        model: YOLO segmentation model (e.g., YOLOv11)
        image_dir: directory containing input images
        image_numbers: list of image numbers (e.g., [1, 2, 3])
        conf: confidence threshold for predictions
        cols: number of columns in the subplot grid
        figsize: overall figure size
    """
    image_dir = Path(image_dir)
    rows = int(np.ceil(len(image_numbers) / cols))
    plt.figure(figsize=figsize)

    for i, num in enumerate(image_numbers):
        filename = f"{num:04d}.png"
        img_path = image_dir / filename

        results = model.predict(source=str(img_path), conf=conf, verbose=False, show=False, save=False)
        if not results or results[0].masks is None:
            print(f"[Warning] No mask found in {filename}")
            continue

        # Read image
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        overlay = img.copy()

        # Draw all predicted mask contours
        for mask_xy in results[0].masks.xy:
            contour = mask_xy.astype(np.int32)
            cv2.polylines(overlay, [contour], isClosed=True, color=(0, 255, 0), thickness=7)

        if results[0].boxes is not None:
            boxes_xywh = results[0].boxes.xywh.cpu().numpy()

            for box in boxes_xywh:
                x_c, y_c, w, h = box

                # Center of the ellipse
                center = (int(x_c), int(y_c))

                # Axes: OpenCV expects radii (half of width/height)
                axes = (int(w / 2), int(h / 2))

                # Draw the ellipse
                cv2.ellipse(overlay, center, axes, 0, 0, 360, (255, 0, 0), 10)

        # Blend overlay for nicer visualization
        blended = cv2.addWeighted(img, 0.7, overlay, 0.3, 0)

        # Plot
        plt.subplot(rows, cols, i + 1)
        plt.imshow(blended)
        plt.title(filename)
        plt.axis("off")

    plt.tight_layout()
    plt.savefig('papilla_pred.png')
    plt.show()

In [ ]:
trained_model = YOLO("runs/train/optic_disc_segmentation2/weights/best.pt")
show_optic_disc_shapes(trained_model, "data/optic_disc/train/images", image_numbers=range(700, 750), cols=3, figsize=(20, 100))

In [ ]:
def get_polygon_from_txt(txt_path, img_w, img_h):
    """
    Reads a YOLO format txt file and converts normalized coordinates
    to absolute pixel coordinates.
    """
    try:
        with open(txt_path, 'r') as f:
            lines = f.readlines()
    except FileNotFoundError:
        return None

    polygons = []
    for line in lines:
        data = list(map(float, line.strip().split()))
        # data[0] is class_id, skip it
        coords = data[1:]

        poly = []
        for i in range(0, len(coords), 2):
            x = int(coords[i] * img_w)
            y = int(coords[i+1] * img_h)
            poly.append([x, y])
        polygons.append(np.array(poly, np.int32))

    return polygons

def calculate_mask_dice(gt_polys, pred_masks, img_h, img_w):
    """
    Calculates Dice Coefficient between Ground Truth polygons and Predicted masks.
    Formula: 2 * Intersection / (Area_GT + Area_Pred)
    """
    # Create blank binary masks
    mask_gt = np.zeros((img_h, img_w), dtype=np.uint8)
    mask_pred = np.zeros((img_h, img_w), dtype=np.uint8)

    # Draw Ground Truth
    if gt_polys:
        cv2.fillPoly(mask_gt, gt_polys, 1)

    # Draw Prediction
    if pred_masks is not None:
        for xy in pred_masks.xy:
            if len(xy) > 0:
                contour = xy.astype(np.int32)
                cv2.fillPoly(mask_pred, [contour], 1)

    # Calculate Dice
    intersection = np.logical_and(mask_gt, mask_pred).sum()
    area_gt = mask_gt.sum()
    area_pred = mask_pred.sum()

    total_area = area_gt + area_pred

    if total_area == 0:
        # Both masks are empty, technically a perfect match (no object present and none detected)
        return 1.0

    dice_score = (2 * intersection) / total_area
    return dice_score

def show_worst_predictions(model, image_dir, label_dir, search_range, num_worst=50, cols=5, figsize=(20, 100)):
    image_dir = Path(image_dir)
    label_dir = Path(label_dir)

    scored_results = []

    print(f"Comparing predictions vs Ground Truth (Dice Score) for {len(search_range)} images...")

    # Scan and score
    for num in tqdm(search_range):
        filename_img = f"{num:04d}.png"
        filename_txt = f"{num:04d}.txt"

        img_path = image_dir / filename_img
        txt_path = label_dir / filename_txt

        if not img_path.exists():
            continue

        img0 = cv2.imread(str(img_path))
        if img0 is None: continue
        h, w = img0.shape[:2]

        gt_polys = get_polygon_from_txt(txt_path, w, h)

        results = model.predict(source=str(img_path), verbose=False, save=False)
        pred_masks = results[0].masks

        # If GT file is missing, we skip
        if gt_polys is None:
            continue

        dice = calculate_mask_dice(gt_polys, pred_masks, h, w)
        scored_results.append((dice, filename_img, img_path, gt_polys))

    # Sort (Lowest Dice = Worst)
    scored_results.sort(key=lambda x: x[0])
    worst_images = scored_results[:num_worst]

    # Visualise
    print(f"Displaying {len(worst_images)} worst predictions based on Dice Coefficient...")

    rows = int(np.ceil(len(worst_images) / cols))
    plt.figure(figsize=figsize)

    for i, (dice, filename, img_path, gt_polys) in enumerate(worst_images):

        # Re-predict for visualization
        results = model.predict(source=str(img_path), verbose=False, save=False)

        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        overlay = img.copy()

        # Draw GROUND TRUTH in BLUE (0, 0, 255)
        if gt_polys:
            cv2.polylines(overlay, gt_polys, isClosed=True, color=(0, 0, 255), thickness=3)

        # Draw PREDICTION in GREEN (0, 255, 0)
        if results[0].masks is not None:
            for mask_xy in results[0].masks.xy:
                if len(mask_xy) > 0:
                    contour = mask_xy.astype(np.int32)
                    cv2.polylines(overlay, [contour], isClosed=True, color=(0, 255, 0), thickness=2)

        blended = cv2.addWeighted(img, 0.6, overlay, 0.4, 0)

        plt.subplot(rows, cols, i + 1)
        plt.imshow(blended)
        plt.title(f"{filename}\nDice: {dice:.2f}")
        plt.axis("off")

        if i == 0:
            plt.xlabel("Blue: Ground Truth\nGreen: Prediction")

    plt.tight_layout()
    plt.show()

In [ ]:
trained_model = YOLO("runs/train/optic_disc_segmentation2/weights/best.pt")

show_worst_predictions(
    trained_model,
    image_dir="data/optic_disc/train",
    label_dir="data/optic_disc/labels",
    search_range=range(1, 806),
    num_worst=50,
    cols=3,
    figsize=(20, 100)
)

In [ ]:
def pred_optic_disc(
    model,
    image_dir_path,
    label_dir_path,
    images,
    figsize=(10, 10),
    save_image=None,
    WIDTH=4,
    pred=True,
    zoom=True
):
    image_dir_path = Path(image_dir_path)
    label_dir_path = Path(label_dir_path)

    fig, axes = plt.subplots(2, 4, figsize=figsize)
    axes = axes.flatten()

    plt.subplots_adjust(left=0, right=1, top=1, bottom=0, wspace=0, hspace=0)

    for i, image in enumerate(images):
        filename_img = f"{image:04d}.png"
        filename_txt = f"{image:04d}.txt"

        img_path = image_dir_path / filename_img
        txt_path = label_dir_path / filename_txt

        results = model.predict(source=str(img_path), verbose=False, save=False)

        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        overlay = img.copy()

        center_x, center_y = w // 2, h // 2  # Default center if no detection

        if pred:
            # Prediction masks (Green)
            if results[0].masks is not None:
                for mask_xy in results[0].masks.xy:
                    contour = mask_xy.astype(np.int32)
                    cv2.polylines(overlay, [contour], True, (0, 255, 0), WIDTH)

            # Ground truth (Blue)
            gt_polys = get_polygon_from_txt(txt_path, w, h)
            if gt_polys:
                cv2.polylines(overlay, gt_polys, True, (20, 20, 255), WIDTH)

            # Ellipses (Red)
            if results[0].boxes is not None and len(results[0].boxes) > 0:
                boxes = results[0].boxes.xywh.cpu().numpy()
                # We take the first detection to center the zoom
                center_x, center_y, bw, bh = boxes[0]

                for x_c, y_c, bw, bh in boxes:
                    center = (int(x_c), int(y_c))
                    axes_ = (int(bw / 2), int(bh / 2))
                    cv2.ellipse(overlay, center, axes_, 0, 0, 360, (255, 0, 0), WIDTH)

        blended = cv2.addWeighted(img, 0.3, overlay, 0.7, 0)

        if zoom:
            # 25% area means 50% width and 50% height
            zoom_w = int(w * 0.5)
            zoom_h = int(h * 0.5)

            # Calculate top-left corner coordinates
            x1 = int(center_x - zoom_w // 2)
            y1 = int(center_y - zoom_h // 2)

            # Adjust coordinates to keep the crop inside image boundaries
            x1 = max(0, min(x1, w - zoom_w))
            y1 = max(0, min(y1, h - zoom_h))
            x2 = x1 + zoom_w
            y2 = y1 + zoom_h

            # Crop the blended image
            display_img = blended[y1:y2, x1:x2]
        else:
            display_img = blended

        axes[i].imshow(display_img, aspect="auto")
        axes[i].axis("off")

    if save_image is not None:
        fig.savefig(
            save_image,
            dpi=300,
            bbox_inches="tight",
            pad_inches=0,
            facecolor="none"
        )

    plt.show()
    plt.close(fig)

In [ ]:
trained_model = YOLO("runs/train/optic_disc_segmentation2/weights/best.pt")
pred_optic_disc(
    trained_model,
    "data/optic_disc/val/images",
    "data/optic_disc/val/labels",
    images=[100],
    figsize=(20,10),
    save_image=None,
    WIDTH=6,
    pred=True,
)

In [ ]:
def analyze_vh_ratio(model_path, img_dir):
    model = YOLO(model_path)
    img_dir = Path(img_dir)
    img_files = list(img_dir.glob("*.png"))

    vh_ratios = []

    print(f"Analyzing V/H ratios for {len(img_files)} images...")

    for img_path in tqdm(img_files):
        results = model.predict(source=str(img_path), verbose=False)[0]

        if results.boxes is not None and len(results.boxes) > 0:
            box = results.boxes[0].xywh.cpu().numpy()[0]

            bw = box[2] # Width (Horizontal Diameter)
            bh = box[3] # Height (Vertical Diameter)

            if bw > 0:
                ratio = bh / bw
                vh_ratios.append(ratio)

    # Calculate statistics
    if vh_ratios:
        mean_ratio = np.mean(vh_ratios)
        std_ratio = np.std(vh_ratios)

        print("\n" + "="*45)
        print("      OPTIC DISC ANATOMICAL ANALYSIS      ")
        print("="*45)
        print(f"Mean V/H Ratio:     {mean_ratio:.4f} ± {std_ratio:.4f}")
        print(f"Sample size (n):    {len(vh_ratios)}")
        print("-" * 45)

        if mean_ratio > 1.0:
            print(f"Observation: The disc is vertically oval (Height > Width).")
        else:
            print(f"Observation: The disc is horizontally oval (Width > Height).")

        percentage_diff = (mean_ratio - 1.0) * 100
        print(f"Vertical elongation: {percentage_diff:.2f}%")
        print("="*45)
    else:
        print("No detections were made to calculate ratios.")

# Exec
analyze_vh_ratio(
    model_path="runs/train/optic_disc_segmentation2/weights/best.pt",
    #img_dir="data/optic_disc/train"
    img_dir="new_images/pngs"
)

In [ ]:
def calculate_repeatability_with_debug(model_path, img_dir, output_dir="debug_frames"):
    """
    Runs inference on a folder of images (same eye), calculates stats,
    and saves every image with the detected box drawn to 'output_dir'.
    """
    model = YOLO(model_path)
    img_dir = Path(img_dir)
    img_files = list(img_dir.glob("*.png"))

    # Create the output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    lengths_px = []

    print(f"Processing {len(img_files)} frames...")
    print(f"Visualized images will be saved to: {os.path.abspath(output_dir)}")

    for img_path in tqdm(img_files):
        results = model.predict(source=str(img_path), verbose=False)[0]

        # Visualize & Save (Draws the box on the image)
        # results.plot() creates a BGR numpy array of the image with boxes/labels
        annotated_frame = results.plot(conf=True, labels=True)

        # Construct output path
        save_path = os.path.join(output_dir, img_path.name)
        cv2.imwrite(save_path, annotated_frame)

        # Collect Data
        if results.boxes and len(results.boxes) > 0:
            # Extract box height (4th element of xywh)
            box = results.boxes[0].xywh.cpu().numpy()[0]
            _, _, _, bh = box
            lengths_px.append(bh)
        else:
            # If no box is found
            print(f"\n[WARNING] No detection in: {img_path.name}")

    if len(lengths_px) < 2:
        print("Error: Not enough detections to calculate statistics.")
        return

    # Statistics
    mean_l = np.mean(lengths_px)
    std_l = np.std(lengths_px)
    cov = (std_l / mean_l) * 100

    print("\n" + "="*40)
    print(f"--- REPEATABILITY RESULTS ---")
    print(f"Output Folder:  {output_dir}")
    print(f"N (frames):     {len(lengths_px)}")
    print(f"Mean Height:    {mean_l:.2f} px")
    print(f"Std Dev:        {std_l:.2f} px")
    print(f"CoV (Precision): {cov:.3f}%")
    print("="*40)

calculate_repeatability_with_debug(
    model_path="runs/train/optic_disc_segmentation2/weights/best.pt",
    img_dir="same"
)

In [ ]:
def get_binary_mask_from_polygon(poly_coords, img_h, img_w):
    """Creates a binary mask from polygon coordinates."""
    mask = np.zeros((img_h, img_w), dtype=np.uint8)
    pts = np.array(poly_coords, dtype=np.int32).reshape((-1, 1, 2))
    cv2.fillPoly(mask, [pts], 1)
    return mask

def get_binary_mask_from_ellipse(box_xywh, img_h, img_w):
    """
    Creates a binary mask of an axis-aligned ellipse based on the bounding box.
    This matches your calibration logic (using box height/width).
    """
    mask = np.zeros((img_h, img_w), dtype=np.uint8)
    x_c, y_c, w, h = box_xywh

    # Ellipse parameters
    center = (int(x_c), int(y_c))
    axes = (int(w / 2), int(h / 2)) # Half-axes

    # Draw filled ellipse (angle=0 because we used xywh box)
    cv2.ellipse(mask, center, axes, 0, 0, 360, 1, -1)
    return mask

def get_binary_mask_from_fitted_ellipse(mask_points, img_h, img_w):
    """
    Alternative: Uses cv2.fitEllipse to get a rotated ellipse from the raw mask points.
    Use this if your "regularization" allows rotation.
    """
    mask = np.zeros((img_h, img_w), dtype=np.uint8)
    if len(mask_points) < 5: return mask # Need 5 points to fit ellipse

    pts = np.array(mask_points, dtype=np.int32)
    try:
        # Returns ((x,y), (a,b), angle)
        ellipse = cv2.fitEllipse(pts)
        cv2.ellipse(mask, ellipse, 1, -1)
    except:
        return mask
    return mask

def calculate_mask_metrics(mask_gt, mask_pred):
    """Calculates IoU and Dice for binary masks."""
    intersection = np.logical_and(mask_gt, mask_pred).sum()
    union = np.logical_or(mask_gt, mask_pred).sum()
    area_gt = mask_gt.sum()
    area_pred = mask_pred.sum()

    iou = intersection / (union + 1e-6)
    dice = (2 * intersection) / (area_gt + area_pred + 1e-6)
    return iou, dice

def evaluate_ellipse_performance(model_path, images_dir, labels_dir):
    model = YOLO(model_path)
    img_files = list(Path(images_dir).glob("*.png"))

    ious = []
    dices = []
    failures = 0
    missing = 0
    total_imgs = 0

    print(f"Evaluating Ellipse-Regularized performance on {len(img_files)} images...")

    for img_path in tqdm(img_files):
        # Load Image Info
        img = cv2.imread(str(img_path))
        if img is None: continue
        h_img, w_img = img.shape[:2]

        # Get Ground Truth Mask
        label_path = Path(labels_dir) / f"{img_path.stem}.txt"
        if not label_path.exists(): continue
        total_imgs += 1

        with open(label_path, 'r') as f:
            lines = f.readlines()
            if not lines: continue
            data = list(map(float, lines[0].strip().split()))

            # Convert label to pixels
            if len(data) > 5: # Polygon
                coords = data[1:]
                poly_pts = []
                for i in range(0, len(coords), 2):
                    poly_pts.append([coords[i] * w_img, coords[i+1] * h_img])
                mask_gt = get_binary_mask_from_polygon(poly_pts, h_img, w_img)
            else: # Box (fallback)
                _, x, y, w, h = data
                continue

        # Model Prediction
        results = model.predict(source=str(img_path), verbose=False)[0]

        if results.boxes and len(results.boxes) > 0:
            box = results.boxes[0].xywh.cpu().numpy()[0] # x_c, y_c, w, h (pixels)
            mask_ellipse = get_binary_mask_from_ellipse(box, h_img, w_img)

            # Calculate Metrics
            iou, dice = calculate_mask_metrics(mask_gt, mask_ellipse)

            ious.append(iou)
            dices.append(dice)

            if iou < 0.5:
                failures += 1
        else:
            missing += 1

    # Results
    print("\n" + "="*50)
    print("SECTION 3.1: ELLIPSE SEGMENTATION PERFORMANCE")
    print("-" * 50)
    print(f"Total Images: {total_imgs}")
    print(f"Mean IoU:  {np.mean(ious):.3f} ± {np.std(ious):.3f}")
    print(f"Mean Dice: {np.mean(dices):.3f} ± {np.std(dices):.3f}")
    print(f"Failure Rate: {failures}/{total_imgs} ({failures/total_imgs*100:.2f}%)")
    print(f"Missing Rate: {missing}/{total_imgs} ({missing/total_imgs*100:.2f}%)")
    print("="*50)

evaluate_ellipse_performance(
    model_path="runs/train/optic_disc_segmentation2/weights/best.pt",
    images_dir="data/optic_disc/train/images",
    labels_dir="data/optic_disc/labels"
)

In [ ]:
def calculate_calibration_stats(model_path, img_dir, L_prior_mm=1.921):
    model = YOLO(model_path)
    img_files = list(Path(img_dir).glob("*.png"))

    # Storage
    l_disc_px = []      # Height of disc in pixels
    p_disc_um = []      # Pixel pitch in microns
    Lx_mm = []          # Image width in mm
    Ly_mm = []          # Image height in mm
    ratios_vh = []      # Vertical/Horizontal ratio (Anatomical check)

    print(f"Calculating calibration metrics for {len(img_files)} images...")

    for img_path in tqdm(img_files):
        img = cv2.imread(str(img_path))
        orig_h, orig_w = img.shape[:2]

        # Predict
        results = model.predict(source=str(img_path), verbose=False)[0]

        if results.boxes and len(results.boxes) > 0:
            box = results.boxes[0].xywh.cpu().numpy()[0]
            bw, bh = box[2], box[3] # Width and Height in pixels (model output is scaled to original)

            # l_disc (pixels) = Height of box
            l_disc = bh

            # Pixel Pitch (mm/px) = Prior / l_disc
            pitch_mm = L_prior_mm / l_disc

            # Lateral Extent (mm)
            width_mm = orig_w * pitch_mm
            height_mm = orig_h * pitch_mm

            # Anatomical Ratio (Vertical / Horizontal)
            # If > 1, it is vertically elongated
            vh_ratio = bh / bw

            # Store
            l_disc_px.append(l_disc)
            p_disc_um.append(pitch_mm * 1000) # Convert to um
            Lx_mm.append(width_mm)
            Ly_mm.append(height_mm)
            ratios_vh.append(vh_ratio)

    # Helper for Mean, SD and CoV
    def stats(data):
        m = np.mean(data)
        s = np.std(data)
        cov = (s/m)*100 if m != 0 else 0
        return m, s, cov

    m_l, s_l, c_l = stats(l_disc_px)
    m_p, s_p, c_p = stats(p_disc_um)
    m_lx, s_lx, c_lx = stats(Lx_mm)
    m_vh, s_vh, c_vh = stats(ratios_vh)

    n_total = len(img_files)
    n_valid = len(l_disc_px)

    print("\n" + "="*50)
    print("SECTION 3.2: PIXEL PITCH & EXTENT")
    print(f"Processed: {n_valid} / {n_total} (Excluded: {n_total - n_valid})")
    print("-" * 50)
    print(f"{'Metric':<25} | {'Mean':<8} | {'SD':<8} | {'CoV (%)':<8}")
    print("-" * 50)
    print(f"{'Disc Height (px)':<25} | {m_l:<8.2f} | {s_l:<8.2f} | {c_l:<8.2f}")
    print(f"{'Pixel Pitch (um/px)':<25} | {m_p:<8.2f} | {s_p:<8.2f} | {c_p:<8.2f}")
    print(f"{'FOV Width (mm)':<25} | {m_lx:<8.2f} | {s_lx:<8.2f} | {c_lx:<8.2f}")
    print(f"{'V/H Ratio':<25} | {m_vh:<8.3f} | {s_vh:<8.3f} | {c_vh:<8.2f}")
    print("="*50)

calculate_calibration_stats(
    model_path="runs/train/optic_disc_segmentation2/weights/best.pt",
    img_dir="data/optic_disc/train/images"
)

In [ ]:
def add_physical_scale(image_path, model, L_disc_prior=1.92, scale_len_mm=1.0):
    """
    Detects the optic disc, calculates pixel pitch, and adds a scale bar.
    """
    # Load Image
    img = cv2.imread(image_path)
    if img is None: return None
    orig_h, orig_w = img.shape[:2]

    # Run Model
    results = model.predict(source=image_path, verbose=False)[0]

    # Check for valid detection (QC Gate: Confidence > 0.7)
    if results.boxes is not None and len(results.boxes) > 0 and results.boxes[0].conf > 0.7:
        box = results.boxes[0].xywh.cpu().numpy()[0]
        _, _, bw, bh = box  # width and height in pixels (auto-scaled by Ultralytics)

        # Calculate Pixel Pitch
        ell_disc_px = bh
        pixel_pitch_mm_per_px = L_disc_prior / ell_disc_px
        #pixel_pitch_mm_per_px = 9.99 / 1000

        # Calculate length of the scale bar in pixels
        bar_len_px = int(scale_len_mm / pixel_pitch_mm_per_px)

        # DRAWING SETTINGS
        padding = 50
        thickness = 6
        color = (255, 255, 255) # White
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 1.2

        # Draw the line
        y = orig_h - padding
        half_t = thickness // 2

        top_left = (padding, y - half_t)
        bottom_right = (padding + bar_len_px, y + half_t)

        cv2.rectangle(img, top_left, bottom_right, color, thickness=-1)

        # Draw the text (e.g., "1 mm")
        text = f"{scale_len_mm} mm"
        text_size = cv2.getTextSize(text, font, font_scale, 2)[0]
        text_x = padding + (bar_len_px // 2) - (text_size[0] // 2)
        text_y = orig_h - padding - 15
        cv2.putText(img, text, (text_x, text_y), font, font_scale, color, 2, cv2.LINE_AA)

        print(f"Scale added: {scale_len_mm}mm = {bar_len_px}px (Pitch: {pixel_pitch_mm_per_px*1000:.2f} um/px)")
    else:
        # If detection fails, add a warning label instead
        cv2.putText(img, "UNCALIBRATED", (50, orig_h - 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
        print("Warning: Optic disc not detected. Image uncalibrated.")

    return img

model = YOLO("runs/train/optic_disc_segmentation2/weights/best.pt")
calibrated_img = add_physical_scale("data/optic_disc/val/images/0033.png", model)

if calibrated_img is not None:
    cv2.imwrite("calibrated_output.png", calibrated_img)




In [ ]:
def put_latex_text(img, text, center_x, center_y, font_size=20, color=(0, 0, 255)):
    """
    Renders LaTeX text with a black outline using Matplotlib
    and overlays it onto an OpenCV image.
    """
    # Create a Matplotlib figure to render the text
    fig, ax = plt.subplots()
    fig.patch.set_alpha(0)  # Transparent background
    ax.axis('off')

    # Convert BGR (OpenCV) to RGB (Matplotlib) 0-1 scale
    text_color = [c/255.0 for c in color[::-1]]

    # Place text
    t = ax.text(0.5, 0.5, text, fontsize=font_size, color=text_color,
                ha='center', va='center', fontname='DejaVu Sans')

    # Linewidth controls the thickness of the black border
    t.set_path_effects([path_effects.withStroke(linewidth=4, foreground='black')])

    # Render the canvas to a buffer
    io_buf = io.BytesIO()
    fig.savefig(io_buf, format='png', bbox_inches='tight', pad_inches=0.05, transparent=True, dpi=100)
    io_buf.seek(0)

    # Load buffer as an OpenCV image (BGRA)
    latex_img = cv2.imdecode(np.frombuffer(io_buf.getvalue(), np.uint8), cv2.IMREAD_UNCHANGED)

    # Close figure to free memory
    plt.close(fig)

    if latex_img is None: return img

    # Overlay logic (Alpha Blending)
    h_text, w_text = latex_img.shape[:2]
    top_left_x = int(center_x - w_text // 2)
    top_left_y = int(center_y - h_text // 2)

    h_img, w_img = img.shape[:2]

    # Clip coordinates
    x1 = max(0, top_left_x)
    y1 = max(0, top_left_y)
    x2 = min(w_img, top_left_x + w_text)
    y2 = min(h_img, top_left_y + h_text)

    # Define ROI in text image
    tx1 = x1 - top_left_x
    ty1 = y1 - top_left_y
    tx2 = tx1 + (x2 - x1)
    ty2 = ty1 + (y2 - y1)

    if x2 <= x1 or y2 <= y1: return img

    # Extract Alpha mask and Text RGB
    overlay_alpha = latex_img[ty1:ty2, tx1:tx2, 3] / 255.0
    overlay_rgb = latex_img[ty1:ty2, tx1:tx2, 0:3]

    # Get background ROI
    bg_roi = img[y1:y2, x1:x2]

    # Blend
    for c in range(3):
        bg_roi[:, :, c] = (overlay_alpha * overlay_rgb[:, :, c] +
                           (1.0 - overlay_alpha) * bg_roi[:, :, c])

    img[y1:y2, x1:x2] = bg_roi

    return img

def process_and_annotate(image_path, model, L_disc_prior=1.921, scale_len_mm=1.0):
    img = cv2.imread(str(image_path))
    if img is None: return None
    orig_h, orig_w = img.shape[:2]

    results = model.predict(source=str(image_path), verbose=False)[0]

    if results.boxes and len(results.boxes) > 0:
        box = results.boxes[0].xywh.cpu().numpy()[0]
        cx, cy, w, h = box

        # Draw Ellipse
        cv2.ellipse(img, (int(cx), int(cy)), (int(w/2), int(h/2)),
                    0, 0, 360, (0, 0, 255), 4, cv2.LINE_AA)

        # Draw Scale Bar (White)
        ell_disc_px = h
        pixel_pitch = L_disc_prior / ell_disc_px
        bar_len_px = int(scale_len_mm / pixel_pitch)

        margin_left = 80
        margin_bottom = 80
        padding = 30
        thickness = 6
        y = orig_h - margin_bottom + padding
        half_t = thickness // 2

        top_left = (margin_left, y - half_t)
        bottom_right = (margin_left + bar_len_px, y + half_t)

        cv2.rectangle(img, top_left, bottom_right, (255, 255, 255), thickness=-1)

        # Draw "1 mm" (Standard OpenCV is fine for this)
        text = f"{scale_len_mm} mm"
        font = cv2.QT_FONT_NORMAL
        font_scale = 1.4
        (tw, th), _ = cv2.getTextSize(text, font, font_scale, 2)
        tx = margin_left + (bar_len_px - tw) // 2
        ty = orig_h - margin_bottom

        cv2.putText(img, text, (tx, ty), font, font_scale, (255, 255, 255), 2, cv2.LINE_AA)

        # Draw LaTeX Label for l_disc
        # Position: Left of the disc
        label_x = int(cx) - int(w/2) + 90
        label_y = int(cy) - int(h/2) - 40

        # LaTeX String: \ell_{disc}
        # Note: We pass color in BGR (Red to match ellipse)
        latex_str = fr"$\ell_{{\rm disc}} = {int(h)}$ px"

        put_latex_text(img, latex_str, label_x, label_y, font_size=30, color=(0, 0, 255))

        return img
    else:
        print(f"No detection in {image_path}")
        return img

def create_comparison_figure(model_path, img_path_normal, img_path_large, output_path="fig_calibration_result.png"):
    model = YOLO(model_path)
    img1 = process_and_annotate(img_path_normal, model)
    img2 = process_and_annotate(img_path_large, model)

    if img1 is None or img2 is None: return

    # Normalize heights
    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]

    if h1 != h2:
        scale = h1 / h2
        img2 = cv2.resize(img2, (int(w2*scale), h1))

    separator = np.zeros((h1, 20, 3), dtype=np.uint8)
    combined = np.hstack((img1, separator, img2))

    cv2.imwrite(output_path, combined)
    print(f"Saved comparison figure to {output_path}")

create_comparison_figure(
    "runs/train/optic_disc_segmentation2/weights/best.pt",
    "data/optic_disc/val/images/0155.png",
    "data/optic_disc/val/images/0122.png"
)

# Result for articles

In [ ]:
def predict_optic_disc_with_scale(model_path, image_path, L_disc_prior=1.92, scale_len_mm=1.0):
    model = YOLO(model_path)
    conf = 0.25

    results = model.predict(source=str(image_path), conf=conf, verbose=False, show=False, save=False)
    if not results or results[0].masks is None:
        print(f"[Warning] No mask found in {image_path}")
        return

    # Rester en BGR (natif OpenCV) — pas de conversion RGB ici
    img = cv2.imread(str(image_path))

    if results[0].boxes is not None:
        boxes_xywh = results[0].boxes.xywh.cpu().numpy()
        for box in boxes_xywh:
            x_c, y_c, w, h = box
            center = (int(x_c), int(y_c))
            axes = (int(w / 2), int(h / 2))

            print(f"Largeur du disque optique : {w:.1f} px")
            # Rouge en BGR = (0, 0, 255)
            cv2.ellipse(img, center, axes, 0, 0, 360, (0, 0, 255), 4)
    else:
        print(f"[Warning] No bounding box found in {image_path}")

    # Sauvegarde pixel-perfect, même résolution que l'original
    cv2.imwrite("calibrated_output.png", img)

    # Ajout de l'échelle physique sur cette image sauvegardée correctement
    calibrated_img = add_physical_scale("calibrated_output.png", model, L_disc_prior, scale_len_mm)

    if calibrated_img is not None:
        cv2.imwrite("calibrated_output.png", calibrated_img)
        # Affichage seulement, pas de sauvegarde via plt
        plt.imshow(cv2.cvtColor(calibrated_img, cv2.COLOR_BGR2RGB))
        plt.axis("off")
        plt.show()
    else:
        print("Calibration failed. No scale added.")

    return calibrated_img

predict_optic_disc_with_scale(
    model_path="runs/train/optic_disc_segmentation2/weights/best.pt",
    image_path="data/optic_disc/val/images/0103.png")